# ASSIGNMENT 1 - Question 6 - Going Deeper for Plant-Disease Diagnosis
### Alumno: Martín Andrés Gutiérrez López
Topics Skip Connections, ResNet Architectures

In [5]:
# ==============================================================================
# Question 6: Going Deeper for Plant-Disease Diagnosis (Case: AgroLens)
# Verification of ResNet Mechanics, Gradient Flow, Parameter Counts & Projection
# ==============================================================================

import torch
import torch.nn as nn

BOLD = "\033[1m"
BOLD_BLUE = "\033[1;34m"
RESET = "\033[0m"

# ------------------------------------------------------------------------------
# Part (b): Empirical Verification of ResNet Gradient Flow
# ------------------------------------------------------------------------------
print(f"\n{BOLD_BLUE}")
print("=" * 75)
print("PART (b): GRADIENT FLOW VERIFICATION (dL/dx = dL/dy * (dF/dx + 1))")
print("=" * 75,f"{RESET}")

x = torch.tensor([[2.0]], requires_grad=True)

# Define a residual function F(x) = w * x^2
w = torch.tensor([0.5], requires_grad=True)
F_x = w * (x**2)

# Residual output y = F(x) + x
y = F_x + x

# Assume scalar loss L = 0.5 * (y - 10)^2
loss = 0.5 * (y - 10.0) ** 2
loss.backward()

# Analytical derivation:
# dL/dy = (y - 10)
# dF/dx = 2 * w * x = 2 * 0.5 * 2.0 = 2.0
# dy/dx = dF/dx + 1 = 3.0
# dL/dx = (y - 10) * 3.0
y_val = 0.5 * (2.0**2) + 2.0  # 4.0
dL_dy = y_val - 10.0  # -6.0
expected_grad_x = dL_dy * (2.0 + 1.0)  # -18.0

print(f"Calculated Output y      : {y.item():.4f}")
print(f"PyTorch Autograd dL/dx   : {x.grad.item():.4f}")
print(f"Analytical dL/dx         : {expected_grad_x:.4f}")
print("Verification Status      : EXACT MATCH\n")

# ------------------------------------------------------------------------------
# Part (c): Parameter Calculation: BasicBlock vs BottleneckBlock (256 channels)
# ------------------------------------------------------------------------------
print(f"\n{BOLD_BLUE}")
print("=" * 75)
print("PART (c): PARAMETER COUNTS (256 IN/OUT CHANNELS, NO BIAS)")
print("=" * 75,f"{RESET}")

# 1. Basic Block: two 3x3 convolutions (256 -> 256 -> 256)
class BasicBlock256(nn.Module):

  def __init__(self):
    super().__init__()
    self.conv1 = nn.Conv2d(256, 256, kernel_size=3, stride=1, bias=False)
    self.conv2 = nn.Conv2d(256, 256, kernel_size=3, stride=1, bias=False)

  def forward(self, x):
    return self.conv2(self.conv1(x))


# 2. Bottleneck Block: 1x1 (256->64) -> 3x3 (64->64) -> 1x1 (64->256)
class BottleneckBlock256(nn.Module):

  def __init__(self):
    super().__init__()
    self.conv1 = nn.Conv2d(256, 64, kernel_size=1, stride=1, bias=False)
    self.conv2 = nn.Conv2d(64, 64, kernel_size=3, stride=1, bias=False)
    self.conv3 = nn.Conv2d(64, 256, kernel_size=1, stride=1, bias=False)

  def forward(self, x):
    return self.conv3(self.conv2(self.conv1(x)))


basic_block = BasicBlock256()
bottleneck_block = BottleneckBlock256()

params_basic = sum(p.numel() for p in basic_block.parameters())
params_bottleneck = sum(p.numel() for p in bottleneck_block.parameters())

print(f"(i)  Basic Block Parameters      : {params_basic:,}")
print(f"(ii) Bottleneck Block Parameters : {params_bottleneck:,}")
print(f"Parameter Reduction Ratio        : {params_basic / params_bottleneck:.2f}x")
print(
    f"Parameters Saved                 : {(1 - params_bottleneck / params_basic) * 100:.2f}%\n"
)

# ------------------------------------------------------------------------------
# Part (d): Projection Shortcut Downsampling (128 -> 256, stride = 2)
# ------------------------------------------------------------------------------
print(f"\n{BOLD_BLUE}")
print("=" * 75)
print("PART (d): PROJECTION SHORTCUT DOWNSAMPLING (128 -> 256, Stride = 2)")
print("=" * 75,f"{RESET}")


class DownsampleResidualBlock(nn.Module):

  def __init__(self, in_channels=128, out_channels=256, stride=2):
    super().__init__()
    # Residual branch F(x)
    self.conv1 = nn.Conv2d(
        in_channels,
        out_channels,
        kernel_size=3,
        stride=stride,
        padding=1,
        bias=False,
    )
    self.relu = nn.ReLU(inplace=True)
    self.conv2 = nn.Conv2d(
        out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False
    )

    # Projection shortcut: 1x1 conv with stride 2 to match dimensions
    self.shortcut = nn.Sequential(
        nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=1,
            stride=stride,
            bias=False,
        ),
        nn.BatchNorm2d(out_channels),
    )

  def forward(self, x):
    identity = self.shortcut(x)
    fx = self.conv2(self.relu(self.conv1(x)))
    out = self.relu(fx + identity)
    return fx, identity, out


# Simulate input batch: (Batch=1, Channels=128, Height=56, Width=56)
input_feature_map = torch.randn(1, 128, 56, 56)
downsample_block = DownsampleResidualBlock()

fx_out, identity_out, block_out = downsample_block(input_feature_map)

print(f"Input Shape x                     : {list(input_feature_map.shape)}")
print(f"Main Branch F(x) Shape            : {list(fx_out.shape)}")
print(f"Projection Shortcut W_s(x) Shape  : {list(identity_out.shape)}")
print(f"Final Sum (F(x) + W_s(x)) Shape   : {list(block_out.shape)}")
print("\nConfirmation: Dimensions matched successfully via 1x1 Conv with stride=2.")
print("\n")




PART (b): GRADIENT FLOW VERIFICATION (dL/dx = dL/dy * (dF/dx + 1))
Calculated Output y      : 4.0000
PyTorch Autograd dL/dx   : -18.0000
Analytical dL/dx         : -18.0000
Verification Status      : EXACT MATCH



PART (c): PARAMETER COUNTS (256 IN/OUT CHANNELS, NO BIAS)
(i)  Basic Block Parameters      : 1,179,648
(ii) Bottleneck Block Parameters : 69,632
Parameter Reduction Ratio        : 16.94x
Parameters Saved                 : 94.10%



PART (d): PROJECTION SHORTCUT DOWNSAMPLING (128 -> 256, Stride = 2)
Input Shape x                     : [1, 128, 56, 56]
Main Branch F(x) Shape            : [1, 256, 28, 28]
Projection Shortcut W_s(x) Shape  : [1, 256, 28, 28]
Final Sum (F(x) + W_s(x)) Shape   : [1, 256, 28, 28]

Confirmation: Dimensions matched successfully via 1x1 Conv with stride=2.


